[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/14_model_evaluation/04_choosing_metrics_case_studies.ipynb)

# Bài 04 — Chọn metric: case study, khoảng tin cậy và so sánh model

**🎯 Mục tiêu**
- Từ mục tiêu kinh doanh → metric ML: 4 case study (gian lận, sàng lọc y tế, gợi ý sản phẩm, dự báo nhu cầu).
- **Khoảng tin cậy** cho metric bằng **bootstrap** — "0.83 hay 0.83 ± 0.02?".
- So sánh hai model đúng cách: **paired bootstrap**, McNemar test.
- Checklist báo cáo đánh giá model.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/14_model_evaluation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics as skm
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from ml_course.datasets import make_customer_churn, make_fraud_transactions
from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Từ mục tiêu kinh doanh đến metric

| Bài toán | Câu hỏi thật | Metric "offline" | Metric kinh doanh |
|---|---|---|---|
| **Gian lận thẻ** | Đội kiểm tra xử lý được 200 cảnh báo/ngày — bắt được bao nhiêu gian lận? | **Precision@k / Recall@k** (k = năng lực xử lý), PR-AUC | số tiền gian lận chặn được − chi phí kiểm tra |
| **Sàng lọc ung thư** | Không được bỏ sót quá 5% ca bệnh; tối thiểu số người phải làm xét nghiệm sâu | **Specificity tại recall ≥ 95%** | chi phí/ca phát hiện |
| **Gợi ý sản phẩm** | Người dùng chỉ xem 10 sản phẩm đầu | **Precision@10, NDCG@10** | CTR, doanh thu (A/B test) |
| **Dự báo nhu cầu** | Tổng sai lệch tồn kho; thiếu hàng tốn hơn thừa hàng | **WAPE**, pinball loss ở phân vị cao | chi phí tồn kho + mất doanh thu |
| **Chấm điểm tín dụng** | Xếp hạng rủi ro + xác suất vỡ nợ đúng để định giá | **ROC-AUC/Gini** + **calibration** (Brier) | lợi nhuận danh mục |

### Case: gian lận với năng lực xử lý cố định

In [ ]:
fraud = make_fraud_transactions(n=40000, fraud_rate=0.01, seed=7)
features = ["amount_vnd", "hour", "distance_from_home_km", "is_foreign", "n_tx_last_24h", "card_age_days"]
X = fraud[features].astype(float).assign(amount_vnd=lambda d: np.log(d.amount_vnd))
y = fraud["is_fraud"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0, stratify=y)
s_lr = LogisticRegression(max_iter=2000).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
s_gb = HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr).predict_proba(X_te)[:, 1]


def precision_recall_at_k(y_true, s, k):
    top = np.argsort(-s)[:k]
    tp = y_true[top].sum()
    return tp / k, tp / y_true.sum()


for name, s in [("Logistic Regression", s_lr), ("HistGB", s_gb)]:
    out = [f"ROC-AUC = {skm.roc_auc_score(y_te, s):.3f}", f"PR-AUC = {skm.average_precision_score(y_te, s):.3f}"]
    for k in [100, 200, 500]:
        p, r = precision_recall_at_k(y_te, s, k)
        out.append(f"P@{k} = {p:.2f}, R@{k} = {r:.2f}")
    print(f"{name:20s} " + " | ".join(out))

Nếu đội chỉ xử lý được 200 cảnh báo, metric quyết định là **P@200 / R@200**. Ở đây Logistic Regression thắng ở mọi metric (HistGB với tham số mặc định overfit khi chỉ có ~200 ca gian lận để học), nhưng nói chung thứ hạng theo ROC-AUC và theo P@k **có thể khác nhau** — ROC-AUC tính trên toàn bộ thứ tự, còn đội vận hành chỉ nhìn phần đầu danh sách. Luôn đo metric khớp với quy trình vận hành.

### Case: sàng lọc y tế — specificity tại recall cố định

In [ ]:
def specificity_at_recall(y_true, s, min_recall=0.95):
    fpr, tpr, thr = skm.roc_curve(y_true, s)
    ok = tpr >= min_recall
    i = np.flatnonzero(ok)[0]                              # ngưỡng cao nhất vẫn đạt recall yêu cầu
    return 1 - fpr[i], thr[i]


for name, s in [("Logistic Regression", s_lr), ("HistGB", s_gb)]:
    spec, t = specificity_at_recall(y_te, s, 0.8)
    print(f"{name:20s}: tại recall ≥ 80%, specificity = {spec:.3f} (ngưỡng {t:.4f}) → {int((1 - spec) * (y_te == 0).sum())} báo động nhầm")

## 2. Khoảng tin cậy bằng bootstrap

Một metric trên tập test là **ước lượng** — tập test khác cho số khác. Bootstrap: lấy mẫu có hoàn lại tập test nhiều lần, tính metric mỗi lần → phân phối → khoảng tin cậy (percentile).

In [ ]:
def bootstrap_ci(y_true, s, metric, n_boot=1000, alpha=0.05, seed=0):
    r = np.random.default_rng(seed)
    n = len(y_true)
    stats = []
    for _ in range(n_boot):
        idx = r.integers(0, n, n)
        if y_true[idx].min() == y_true[idx].max():
            continue
        stats.append(metric(y_true[idx], s[idx]))
    return np.quantile(stats, [alpha / 2, 1 - alpha / 2]), np.array(stats)


for name, s in [("Logistic Regression", s_lr), ("HistGB", s_gb)]:
    (lo, hi), _ = bootstrap_ci(y_te, s, skm.average_precision_score)
    print(f"{name:20s} PR-AUC = {skm.average_precision_score(y_te, s):.3f}, 95% CI [{lo:.3f}, {hi:.3f}]")
print(f"(chỉ có {y_te.sum()} giao dịch gian lận trong test → khoảng tin cậy rộng)")

## 3. So sánh hai model: paired bootstrap

Hai khoảng tin cậy chồng nhau **chưa** có nghĩa là không khác biệt — vì hai model được đánh giá trên **cùng** dữ liệu (tương quan). Đúng cách: bootstrap **hiệu** metric trên cùng các mẫu bootstrap.

In [ ]:
def paired_bootstrap_diff(y_true, s_a, s_b, metric, n_boot=1000, seed=0):
    r = np.random.default_rng(seed)
    n = len(y_true)
    diffs = []
    for _ in range(n_boot):
        idx = r.integers(0, n, n)
        if y_true[idx].min() == y_true[idx].max():
            continue
        diffs.append(metric(y_true[idx], s_b[idx]) - metric(y_true[idx], s_a[idx]))
    diffs = np.array(diffs)
    return diffs.mean(), np.quantile(diffs, [0.025, 0.975]), np.mean(diffs <= 0)


for metric_name, metric in [("ROC-AUC", skm.roc_auc_score), ("PR-AUC", skm.average_precision_score)]:
    mean, (lo, hi), p_le0 = paired_bootstrap_diff(y_te, s_lr, s_gb, metric)
    print(f"{metric_name}: HistGB − LR = {mean:+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}] | tỉ lệ bootstrap với hiệu ≤ 0: {p_le0:.3f}")

In [ ]:
_, stats_lr = bootstrap_ci(y_te, s_lr, skm.average_precision_score)
_, stats_gb = bootstrap_ci(y_te, s_gb, skm.average_precision_score)
r = np.random.default_rng(0)
paired_diffs = []
for _ in range(1000):
    idx = r.integers(0, len(y_te), len(y_te))
    if y_te[idx].min() != y_te[idx].max():
        paired_diffs.append(skm.average_precision_score(y_te[idx], s_gb[idx]) - skm.average_precision_score(y_te[idx], s_lr[idx]))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].hist(stats_lr, bins=40, alpha=0.6, label="Logistic Regression")
axes[0].hist(stats_gb, bins=40, alpha=0.6, label="HistGB")
axes[0].set_xlabel("PR-AUC trên từng mẫu bootstrap")
axes[0].set_title("Bootstrap riêng từng model")
axes[0].legend()
axes[1].hist(paired_diffs, bins=40, color="C2", alpha=0.8)
axes[1].axvline(0, color="k", ls="--")
axes[1].set_xlabel("PR-AUC(HistGB) − PR-AUC(LR) trên CÙNG mẫu bootstrap")
axes[1].set_title("Bootstrap ghép cặp: phân phối của hiệu")
plt.tight_layout()
plt.show()
ci_lr, ci_gb, ci_diff = (np.quantile(v, [0.025, 0.975]) for v in (stats_lr, stats_gb, paired_diffs))
overlap = ci_lr[0] <= ci_gb[1] and ci_gb[0] <= ci_lr[1]
print(f"KTC riêng: LR [{ci_lr[0]:.3f}, {ci_lr[1]:.3f}], HistGB [{ci_gb[0]:.3f}, {ci_gb[1]:.3f}] → {'chồng nhau' if overlap else 'tách rời'} | "
      f"KTC của hiệu ghép cặp [{ci_diff[0]:+.3f}, {ci_diff[1]:+.3f}] → {'không chứa 0' if ci_diff[1] < 0 or ci_diff[0] > 0 else 'chứa 0'}")

Nếu khoảng tin cậy của **hiệu** chứa 0 → chưa đủ bằng chứng model nào tốt hơn. Ở đây cả hai khoảng đều **âm và không chứa 0** → Logistic Regression tốt hơn HistGB (mặc định) một cách đáng tin trên dữ liệu này.

### McNemar test (cho dự đoán nhãn)

Chỉ xét các mẫu mà **hai model bất đồng**: $b$ = A đúng B sai, $c$ = A sai B đúng. Nếu hai model tốt như nhau, $b \approx c$. Thống kê $\chi^2 = \frac{(|b - c| - 1)^2}{b + c}$ (1 bậc tự do).

In [ ]:
from scipy.stats import chi2

churn = make_customer_churn(n=6000, seed=4)
Xc = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float)
yc = churn["churn"].to_numpy()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.4, random_state=0, stratify=yc)
pa = LogisticRegression(max_iter=3000).fit(Xc_tr, yc_tr).predict(Xc_te)
pb = HistGradientBoostingClassifier(random_state=0).fit(Xc_tr, yc_tr).predict(Xc_te)
b = np.sum((pa == yc_te) & (pb != yc_te))
c = np.sum((pa != yc_te) & (pb == yc_te))
stat = (abs(b - c) - 1) ** 2 / (b + c)
print(f"accuracy: LR = {np.mean(pa == yc_te):.4f} | HistGB = {np.mean(pb == yc_te):.4f}")
print(f"b (LR đúng, GB sai) = {b}, c (LR sai, GB đúng) = {c} | χ² = {stat:.3f}, p-value = {chi2.sf(stat, 1):.3f}")

p-value > 0.05: chênh lệch accuracy khoảng 1 điểm phần trăm **chưa có ý nghĩa thống kê** — hai model bất đồng trên ~230 khách hàng và mỗi model đúng ở khoảng một nửa số đó. Đừng tuyên bố "model mới tốt hơn" dựa trên chênh lệch nhỏ như vậy.

## 4. Checklist báo cáo đánh giá model

1. **Metric chính** gắn với mục tiêu nghiệp vụ (và lý do chọn); metric phụ để phát hiện tác dụng phụ.
2. **Baseline** (đoán đa số/trung bình, model đơn giản, hệ thống hiện tại).
3. **Khoảng tin cậy** (bootstrap) và kiểm định khi so sánh model.
4. **Ngưỡng** chọn trên validation, theo chi phí/ràng buộc.
5. **Phân tích theo phân khúc** (vùng, nhóm khách hàng, thời gian) — model có công bằng và ổn định không?
6. **Calibration** nếu dùng xác suất.
7. Giao thức chia dữ liệu (thời gian, nhóm) phản ánh đúng cách model sẽ được dùng (chương 15).
8. Offline metric chỉ là **đại diện** — xác nhận bằng A/B test / theo dõi sau triển khai (chương 40).

## 🧠 Kiểm tra nhanh

1. Đội chăm sóc khách hàng chỉ gọi được 300 khách/tuần để giữ chân. Metric nào phù hợp cho model churn?
2. Model mới có PR-AUC 0.42 so với 0.40 của model cũ. Cần thêm thông tin gì trước khi kết luận?
3. Vì sao bootstrap hiệu metric (paired) đúng hơn so sánh hai khoảng tin cậy riêng?
4. McNemar test dùng những mẫu nào?

## 📌 Tóm tắt

- Metric phải trả lời câu hỏi nghiệp vụ: @k khi năng lực xử lý cố định, specificity@recall cho y tế, NDCG cho gợi ý, WAPE/pinball cho dự báo.
- Báo cáo metric kèm khoảng tin cậy; so sánh model bằng paired bootstrap / McNemar.
- Checklist: baseline, ngưỡng, phân khúc, calibration, giao thức chia dữ liệu, A/B test.

## 📚 Tài liệu tham khảo

- Dietterich (1998) — *Approximate Statistical Tests for Comparing Supervised Classification Learning Algorithms*
- Raschka (2018) — *Model Evaluation, Model Selection, and Algorithm Selection in Machine Learning*
- Kohavi, Tang & Xu (2020) — *Trustworthy Online Controlled Experiments*